In [ ]:
from openff.toolkit import Molecule

from openff.pablo import CCD_RESIDUE_DEFINITION_CACHE, topology_from_pdb
from utils import draw_molecule

%load_ext snakeviz

In [ ]:
from pprint import pprint

# pprint(CCD_RESIDUE_DEFINITION_CACHE._definitions)

# {key: len(value) for key, value in CCD_RESIDUE_DEFINITION_CACHE._definitions.items()}

CCD_RESIDUE_DEFINITION_CACHE

In [ ]:
Molecule.from_smiles(
    "O=C([O-])Cn1cc(cn1)c2ccc(cc2OCC#N)Nc3ccc(c(n3)NC4CCCCC4)C#N"
)

In [ ]:
# %%snakeviz
# top = topology_from_pdb("/home/joshmitchell/Downloads/2p41.pdb")
top = topology_from_pdb(
    "5ap1_prepared.pdb",
    unknown_molecules=[
        Molecule.from_smiles(
            "O=C([O-])Cn1cc(cn1)c2ccc(cc2OCC#N)Nc3ccc(c(n3)NC4CCCCC4)C#N"
        )
    ],
)

In [ ]:
w = top.visualize()
w.add_representation(
    "line",
    # sele="not hydrogen",
)
w

In [ ]:
top.n_atoms

In [ ]:
from pathlib import Path

from openff.pablo._pdb_data import PdbData

data = PdbData.parse_pdb(Path("5ap1_prepared.pdb").read_text().splitlines())

In [ ]:
w = top.visualize()
w.clear_representations()
w.add_licorice(selection="HIS")
w

In [ ]:
top.molecule(129).visualize(backend="nglview")

In [ ]:
from openff.toolkit import Topology

old_top = Topology.from_pdb(
    "5ap1_prepared.pdb",
    unique_molecules=[
        Molecule.from_smiles(
            "O=C([O-])Cn1cc(cn1)c2ccc(cc2OCC#N)Nc3ccc(c(n3)NC4CCCCC4)C#N"
        )
    ],
)

In [ ]:
for new, old in zip(list(top.molecules)[1:], list(old_top.molecules)[1:]):
    assert new.is_isomorphic_with(old)

In [ ]:
for new, old in zip(top.molecules, old_top.molecules):
    assert new.n_atoms == old.n_atoms
    for new_atom, old_atom in zip(new.atoms, old.atoms):
        new_atom_bits = (new_atom.symbol, new_atom.formal_charge, new_atom.name)
        old_atom_bits = (old_atom.symbol, old_atom.formal_charge, old_atom.name)
        if new_atom_bits != old_atom_bits:
            print(new_atom.metadata["residue_name"], new_atom_bits, old_atom_bits)

    new_bonds = {tuple(sorted([bond.atom1_index, bond.atom2_index])): bond for bond in new.bonds}
    old_bonds = {tuple(sorted([bond.atom1_index, bond.atom2_index])): bond for bond in old.bonds}

    assert set(new_bonds.keys()) == set(old_bonds.keys())
    
    for new_bond in new.bonds:
        old_bond = old_bonds[tuple(sorted([new_bond.atom1_index, new_bond.atom2_index]))]
        if new_bond.bond_order != old_bond.bond_order:
            print(
                new_bond.atom1.metadata["residue_name"],
                new_bond.atom1.metadata["chain_id"],
                new_bond.atom1.metadata["res_seq"],
                new_bond.atom2.metadata["residue_name"],
                new_bond.atom2.metadata["chain_id"],
                new_bond.atom2.metadata["res_seq"],
                new_bond,
                new_bond.bond_order,
                old_bond.bond_order,
            )

In [ ]:
assert top.molecule(0).is_isomorphic_with(old_top.molecule(0))